# **Tiling 开发指引**

前面三节已经完成算子逻辑分析。从本节起进入「开发指引」：为使 Dispatch / Combine 可运行，需要明确 host 侧数据准备方式和 Kernel 阶段组织方式。本节是开发指引第 1 节，聚焦 **host 侧 Tiling**。

---

## **工程目录结构**

本实践示例代码核心目录如下：

```
answer/scaffold_01_10/           # 完整脚手架（含量化版与非量化版）
├─ include/                       # 头文件目录
│  ├─ moe_distribute_dispatch.h           # ★ Dispatch 算子核心头文件
│  │                                      #   - DispatchTilingData 结构体定义
│  │                                      #   - SetTilingDataAndCal() 派生量计算
│  │                                      #   - Init / Process 主流程
│  ├─ moe_distribute_dispatch_quant.h     # MX 量化模板（QuantInit/QuantProcess/CopyScalesToOut）
│  ├─ moe_distribute_dispatch_non_quant.h # 非量化版参考（对比理解量化版）
│  ├─ moe_distribute_combine.h            # Combine 算子核心头文件
│  │                                      #   - MoeDistributeCombineShmemTilingData 结构体
│  │                                      #   - Combine Init / Process 流程
│  └─ moe_distribute_comm.h               # 通信公共头（Win区偏移常量、辅助函数）
├─ src/                           # 源文件目录
│  ├─ dispatch_and_combine_final.asc      # ★ Host 主程序 + Kernel 入口
│  │                                      #   - SetDispatchTilingData / SetCombineTilingData
│  │                                      #   - DispatchKernel / CombineKernel 入口
│  ├─ 0_non_quant_naive.asc               # 非量化版主程序（参考对比）
│  ├─ dispatch_and_combine_final.asc      # 量化版主程序
│  └─ utils.h                             # 测试辅助（bin读写、校验）
├─ scripts/                       # 测试脚本
│  ├─ gen_data.py                         # 测试数据生成
│  └─ verify_result.py                    # 精度校验
└─ cmake/                         # 构建配置
   ├─ ascend.cmake                        # Ascend 编译配置
   └─ shmem.cmake                         # shmem 依赖配置
```

**关键文件作用速览**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">文件</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">核心内容</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">本章关联</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moe_distribute_dispatch.h</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Dispatch算子类、TilingData、派生量计算</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">★ 本节重点：TilingData结构体、SetTilingDataAndCal</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moe_distribute_combine.h</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Combine算子类、TilingData</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本节提及：字段差异对比</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>dispatch_and_combine_final.asc</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Host侧Tiling函数、Kernel入口</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">★ 本节重点：SetDispatchTilingData / SetCombineTilingData</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>0_non_quant_naive.asc</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">非量化版Host主程序</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">参考对比</td>
  </tr>
</table>

---

## **查看关键代码文件**

本节涉及的核心代码位于 Host 主程序和算子头文件，可通过以下 cat 命令查看：

In [ ]:
# 查看 Host 主程序中的 Tiling 函数定义
!cat scaffold_02_10/src/dispatch_and_combine_final.asc | head -100

In [ ]:
# 查看 Dispatch TilingData 结构体定义（在头文件中搜索 DispatchTilingData）
!grep -A 15 'struct.*DispatchTilingData' scaffold_02_10/include/moe_distribute_dispatch.h

In [ ]:
# 查看 Combine TilingData 结构体定义
!grep -A 15 'struct.*MoeDistributeCombineShmemTilingData' scaffold_02_10/include/moe_distribute_combine.h

---
**学习目标**：理解 Dispatch / Combine 各自需要哪些 Tiling 字段、字段从哪里来、host 与 kernel 在「派生量」上的分工，并能照样写出最小可跑的 host 端 Tiling 代码。

**本 sample 的 host 形态**：本 sample 是 AscendC kernel-direct demo：

- Tiling 不走 `ge::Tiling` / `aclnnXxxGetWorkspaceSize` 那套注册流程，而是 host 端直接构造一个 `DispatchTilingData` / `MoeDistributeCombineShmemTilingData` 结构体，按值传给 kernel；
- Kernel 用 `__global__ __aicore__ __vector__ void DispatchKernel(...)` 形式定义，host 用 `DispatchKernel<<<AIV_CORE_NUM, nullptr, stream>>>(...)` 直接 launch；

---

## **1. TilingData 字段清单**

Dispatch 与 Combine 各自的 TilingData 是独立结构体，按值（by-value）传入 kernel。

**Dispatch（非量化版本）**：

```cpp
struct alignas(8) DispatchTilingData {
    uint32_t epWorldSize;          // EP 通信域卡数
    uint32_t epRankId;             // 本卡在 EP 域内的 id
    uint32_t moeExpertNum;         // 全集群专家总数
    uint32_t bs;                   // 本卡 batch size token 数
    uint32_t k;                    // 每个 token 选的 topk 专家数
    uint32_t h;                    // hidden size
    uint32_t aivNum;               // 本卡参与的 AIV 核数 = block dim
    uint32_t expertTokenNumsType;  // 0 cumsum 模式 / 1 count 模式
    uint32_t totalWinSizeEp;       // Win 区总大小（字节）
};
```

**Combine**：

```cpp
struct MoeDistributeCombineShmemTilingData {
    uint32_t epWorldSize;
    uint32_t epRankId;
    uint32_t moeExpertNum;
    uint32_t moeExpertPerRankNum;   // 每张卡承载的本地专家数 = moeExpertNum / epWorldSize
    uint32_t globalBs;              // 全局 batch = bs * epWorldSize
    uint32_t bs;
    uint32_t k;
    uint32_t h;
    uint32_t aivNum;
    uint64_t totalWinSize;
};
```

**字段分类**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">类别</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">字段</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">来源</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">通信拓扑</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>epWorldSize</code> / <code>epRankId</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">通信框架初始化阶段已有</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">MoE 配置</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moeExpertNum</code> / <code>moeExpertPerRankNum</code> / <code>k</code> / <code>expertTokenNumsType</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">模型超参</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">数据形状</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>bs</code> / <code>h</code> / <code>globalBs</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">当次输入的张量形状</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">硬件</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aivNum</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">运行时由 <code>PlatformAscendCManager::GetCoreNumAiv()</code> 查得</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win 区预算</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>totalWinSizeEp</code> / <code>totalWinSize</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">调用方申请 <code>aclshmem</code> 时给定</td>
  </tr>
</table>

**实现说明**：Combine 多了 `moeExpertPerRankNum` 与 `globalBs`，并且 `totalWinSize` 是 64-bit。Dispatch 的 `totalWinSizeEp` 是 32-bit。两者实际填同一个 shmem 大小值（demo 中都填 `SHMEM_SPACE_SIZE`），但类型与字段名不同，复制粘贴时不要漏掉。

---

## **2. Host 侧 Tiling 代码（最小可运行示例）**

Host 端 Tiling 函数的核心动作就是 **按值填充结构体**——sample 中并没有复杂派生计算，所有派生量都留给 kernel 端算（见下一节）。

**最小示例（节选自 [src/0_non_quant_naive.asc](./answer/scaffold_01_10/src/0_non_quant_naive.asc)）**：

```cpp
void SetDispatchTilingData(DispatchTilingData& dispatchTilingData,
                            int epWorldSize, int epRankId, int bs,
                            uint32_t aivNum, uint64_t totalWinSizeEp) {
    dispatchTilingData.epWorldSize        = epWorldSize;
    dispatchTilingData.epRankId           = epRankId;
    dispatchTilingData.moeExpertNum       = epWorldSize * 4;  // demo: 每卡 4 本地专家
    dispatchTilingData.bs                 = bs;
    dispatchTilingData.k                  = 8;                 // demo: topk = 8
    dispatchTilingData.h                  = 7168;              // demo: hidden = 7168
    dispatchTilingData.expertTokenNumsType = 1;                // demo: count 模式
    dispatchTilingData.aivNum             = aivNum;
    dispatchTilingData.totalWinSizeEp     = totalWinSizeEp;
}

void SetCombineTilingData(MoeDistributeCombineShmemTilingData& combineTilingData,
                          int epWorldSize, int epRankId, int bs,
                          uint32_t aivNum, uint64_t totalWinSize) {
    combineTilingData.epWorldSize        = epWorldSize;
    combineTilingData.epRankId           = epRankId;
    combineTilingData.moeExpertPerRankNum = 4;
    combineTilingData.moeExpertNum       = epWorldSize * combineTilingData.moeExpertPerRankNum;
    combineTilingData.globalBs           = bs * epWorldSize;
    combineTilingData.bs                 = bs;
    combineTilingData.k                  = 8;
    combineTilingData.h                  = 7168;
    combineTilingData.aivNum             = aivNum;
    combineTilingData.totalWinSize       = totalWinSize;
}
```

**注意**：上面 `moeExpertNum / k / h / moeExpertPerRankNum` 在 demo 中硬编码，是为了让示例代码自包含；真实 op 中这些应当来自调用方传入的 attr 或模型 attr。

---

## **3. Host 不算、Kernel 才算的派生量**

Host TilingData 只承载「外部输入」，所有跟内存布局相关的派生量都在 Kernel 端 `SetTilingDataAndCal` 里就地计算：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">派生量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">Kernel 端计算公式</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">含义</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moeExpertNumPerRank_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moeExpertNum / epWorldSize</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每卡承载多少本地专家</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertIdsCnt_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>bs * k</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">token-专家映射表元素数</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>hOutSize_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>h * sizeof(ExpandXOutType)</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">token 在输出张量上的字节数</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>hOutSizeAlign_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>ceil(h*sizeof(X), 32) * 32 + 32</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">token UB 对齐字节 + 32 B token-flag 头</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>blockCntPerToken_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>ceil(hOutSizeAlign_, 480)</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">一条 token 在 Win 上切多少个 480 B 子块</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>hCommuSize_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>blockCntPerToken_ * 512</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">一条 token 在 Win 上的完整占位</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertPerSizeOnWin_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>bs * hCommuSize_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每个 (本地专家, 源卡) 槽位大小</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>rscvStatusNum_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>epWorldSize * moeExpertNumPerRank_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">接收侧需要等的状态块总数</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aivUsedCumSum_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>clamp(moeExpertNum / 32, 1, aivNum / 2, CUMSUM_MAX_CORE_NUM, rscvStatusNum_)</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">cumsum 阶段占用的核数</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aivUsedAllToAll_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aivNum - aivUsedCumSum_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">dispatch 主流程占用的核数</td>
  </tr>
</table>

**为什么这么分工**：派生量大多依赖编译期常量（`SPLIT_BLOCK_DATA_SIZE = 480`、`SPLIT_BLOCK_SIZE = 512`、`UB_ALIGN = 32`、`CUMSUM_MAX_CORE_NUM = 16` 等），把它们放在 kernel 内的好处：

- 单一事实来源：常量定义在 kernel namespace，host 不需要重复硬编码；
- 跨版本兼容：当 kernel 实现把 `480` 改成别的值，host 完全不用动；
- 当前示例的 TilingData 体积保持小（9~10 个字段），减少 host→device 拷贝开销。

---

## **4. Host 端还需要准备的其它资源**

TilingData 只是 launch 参数之一，还需要：

**1. Win 区**：

```cpp
aclshmemx_init_attr_t attributes;
aclshmemx_uniqueid_t  defaultFlagUid;
test_set_attr(rankId, rankNum, SHMEM_SPACE_SIZE, ipport, defaultFlagUid, &attributes);
ACL_CHECK_WITH_RET(aclshmemx_init_attr(ACLSHMEMX_INIT_WITH_DEFAULT, &attributes), ERROR_LOG("aclshmemx_init_attr failed"), return -1);  // ACL_CHECK_WITH_RET 为错误检查宏：若初始化失败，记录错误日志并返回-
void* shmemSpace = aclshmem_malloc(SHMEM_SPACE_SIZE);   // 普通版
// 或：aclshmem_align(DISPATCH_PACKAGE_SIZE, SHMEM_SPACE_SIZE);  // 512 字节对齐版
```

- `SHMEM_SPACE_SIZE` demo 取 `1024 MiB`，对应 TilingData 里 `totalWinSizeEp / totalWinSize`；
- 这块内存按 [§ 01.07](./01.07_win_memory_layout.ipynb) 介绍：前 1022 MiB Win数据区、后 2 MiB Win状态区，Dispatch / Combine 共用。

**2. Workspace**：Dispatch 需要一段普通 HBM 作为暂存。Demo 取 16 MiB：

```cpp
uint8_t* disaptchWorkspaceGM;
ACL_CHECK(aclrtMalloc(reinterpret_cast<void**>(&disaptchWorkspaceGM),
            16 * 1024 * 1024, ACL_MEM_MALLOC_HUGE_FIRST));
uint8_t* combineWorkspaceGM;
ACL_CHECK(aclrtMalloc(reinterpret_cast<void**>(&combineWorkspaceGM),
            16 * 1024 * 1024, ACL_MEM_MALLOC_HUGE_FIRST));
```

Combine 不需要单独 workspace（已在 Win 区内复用）。

**3. 算子的输入输出张量预估**：按 (bs, h)、(bs, k)、(maxReceivedTokenNum, h) 等形状预分配，其中：

```cpp
size_t localExpertNum    = moeExpertNum / epWorldSize;
size_t maxReceivedTokenNum =
    bs * epWorldSize * std::min<size_t>(k, localExpertNum);
```

`maxReceivedTokenNum` 是本卡可能收到的 token 上限（每张源卡每个本地专家最多收 `min(k, localExpertNum)` 路），用于给 `expandXOut / dynamicScalesOut / assistInfo` 等输出张量分配 buffer。

**4. Launch 参数**：`block dim = aivNum`，不用单独算 tiling：

```cpp
uint32_t aivCoreNum = PlatformAscendCManager::GetInstance()->GetCoreNumAiv();
DispatchKernel<<<aivCoreNum, nullptr, stream>>>(
    shmemSpace, xInDevice, expertIdsDevice,
    expandXOutDevice, dynamicScalesDevice, tokenSrcInfoDevice,
    expertTokenNumsDevice, sendCountsDevice,
    disaptchWorkspaceGM, dispatchTilingData);
CombineKernel<<<aivCoreNum, nullptr, stream>>>(
    shmemSpace, expandXInDevice, expertIdsDevice, tokenSrcInfoDevice,
    sendCountsDevice, expertScalesDevice, xOutDevice,
    combineWorkspaceGM, combineTilingData);
```

**注意**：`TilingData` 是 **按值** 传入（不是指针），AscendC 编译器会自动把它放到合适的位置；不要传指针，也不要在 launch 后修改 host 端的 TilingData 副本。

---

## **5. 实操要点小结**

- **结构体保持小而稳**：TilingData 只承载「外部输入 + 拓扑 + 硬件 + 预算」，不放任何依赖编译期常量的派生量；派生量统一在 kernel 入口的 `SetTilingDataAndCal` 中算一次，写进成员变量供整个 `Process()` 复用。
- **Dispatch / Combine 字段差异要对齐**：两个结构体各自独立，`moeExpertPerRankNum` / `globalBs` 仅 Combine 有；`totalWinSize` 是 `uint64_t` 而 Dispatch 用 `uint32_t totalWinSizeEp`——同一份 shmem 大小、不同字段，慎防错位赋值。
- **shmem 与 workspace 不要混淆**：shmem（Win 区）由 `aclshmem_malloc` 分配、所有卡之间对称，承担跨卡通信；workspace（仅 dispatch 需要）由 `aclrtMalloc` 分配、单卡私有，承担本卡内的临时存放。两者类型不同、生命周期不同、传入参数也不同。
- **block dim 等于 aivNum**：必须用 `PlatformAscendCManager::GetCoreNumAiv()` 动态查询，且把这个值同时写进 `tilingData.aivNum` 与 launch 配置，二者必须一致——kernel 内会用 `aivNum_` 切分核分工，写错会导致核越界。
- **maxReceivedTokenNum 是 host 自己算**：`bs × epWorldSize × min(k, localExpertNum)`；这只是 I/O 张量大小预估，不进 TilingData，仅在 host 分配 `expandXOut` 等输出时使用。

---

**导航**：[← 上一章：Win区布局](02.07_win_memory_layout.ipynb) | [返回课程主页](02.01_chapter_intro.ipynb) | [下一章：Kernel Stage开发指引 →](02.09_kernel_stage_guide.ipynb)

---